# EDA: play calling (action events, PPP and EPV gain)

Tables only, from `analytics.playcalling.eda`. Inputs come from `analytics/playcalling/synthetic.py` on the shared `analytics.fixtures` games (or the lane's local generator); on either, every number below is **fixture output, not a result**. Re-run with `inputs = validate.load_data_root(DATA_ROOT, OUT_ROOT)` on the Mac for real tables.

Every quantity here is an association: the action type is chosen by the offence against a defence and a lineup, so a PPP difference between action types is not the effect of calling the play.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'analytics').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from analytics.playcalling import validate, features, eda

inputs = validate.fixture_inputs(seed=0, n_games=8, n_possessions=40, heldout_every=4)   # fixture output, not a result
inputs, vlog = validate.validate_inputs(inputs)
display(Markdown(validate.format_report(vlog)))

In [ ]:
table, build_log = features.build_action_table(inputs['frames'], inputs['possessions'], inputs['actions'], inputs['screens'],
                                                 inputs['coverage_predictions'], inputs['epv_states'], inputs['heldout_games'])
print(build_log)
train = table[~table['heldout']]      # held-out rows are never summarised here
print(len(table), 'actions,', len(train), 'training rows')

## Action counts (by label source)

In [ ]:
display(Markdown(eda.to_markdown(eda.action_counts(train))))

## Coverage counts (by coverage source)

In [ ]:
display(Markdown(eda.to_markdown(eda.coverage_counts(train))))

## PPP distribution per action type

Shares of 0, 1, 2 and 3+ points of the containing possession, the mean, and the mean EPV gain across the action when epv_states exist.

In [ ]:
display(Markdown(eda.to_markdown(eda.ppp_distribution(train))))

## Lineup counts

How many distinct five-man lineups the rows carry and how concentrated they are; `lineups_with_300_plus` says how many could pass the 300 gate on their own.

In [ ]:
display(Markdown(eda.to_markdown(eda.lineup_counts(train))))

## Shuffled-action check

The spread (max minus min) of mean PPP across action types, observed against the spread when action labels are shuffled within each game. A small `share_shuffled_ge_observed` means the labels carry information about PPP beyond game-to-game differences. It is a check of association, not of a causal effect.

In [ ]:
display(Markdown(eda.to_markdown(eda.shuffled_action_check(train, n_shuffles=200, seed=0), nd=4)))

## Leakage guard

The feature frame holds no outcome: `features.features` calls `analytics.schema.assert_time_forward` and refuses `ppp` and `epv_gain`.

In [ ]:
print(features.features(train).columns.tolist())